# Module 01 — Mathematical & Programming Foundations## 01-09: Calculus & Optimization Foundations**Objective:** Build the calculus and optimization toolkit for machinelearning — numerical differentiation, gradients, the chain rule, Jacobians,and gradient descent variants — from scratch, then verify against PyTorchautograd.**Prerequisites:** 01-01 (Python, NumPy & Tensor Speed), 01-06 (Linear Algebra for ML), 01-08 (Information Theory for ML)

---## Part 0 — Setup & PrerequisitesCalculus is the engine behind learning in neural networks. Every time amodel updates its weights, it's computing derivatives of a loss functionwith respect to parameters and taking a step in the direction of steepestdescent.This notebook covers:- **Numerical differentiation** — finite differences and their limitations- **Partial derivatives and gradients** — navigating multi-dimensional loss landscapes- **The chain rule** — the foundation of backpropagation- **Jacobians and Hessians** — higher-order derivative structures- **Gradient descent** — vanilla, momentum, and adaptive learning rates- **Convergence analysis** — learning rate effects, saddle points, local minimaWe implement everything from scratch with NumPy, then verify againstPyTorch's autograd.**Prerequisites:** 01-01, 01-06, 01-08

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
FIGSIZE = (10, 6)
LEARNING_RATE = 1e-2
NUM_ITERATIONS = 200
COLORS = {
    'blue': '#1E88E5',
    'red': '#E53935',
    'green': '#43A047',
    'orange': '#FF9800',
    'purple': '#9C27B0',
    'teal': '#00897B',
    'gray': '#757575',
}
COLOR_LIST = list(COLORS.values())

### Data LoadingWe use the California Housing dataset for regression-based optimizationdemonstrations.

In [ ]:
# California Housing (for linear regression optimization)
housing = fetch_california_housing()
X_raw = housing.data.astype(np.float64)
y_raw = housing.target.astype(np.float64)

# Use first 2 features for visualization, all features for full demo
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=SEED,
)
print(f'California Housing: X_train={X_train.shape}, X_test={X_test.shape}')
print(f'Target range: [{y_raw.min():.2f}, {y_raw.max():.2f}]')

# Small 2D subset for surface visualizations
X_2d = X_scaled[:, :2]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_raw, test_size=0.2, random_state=SEED,
)
print(f'2D subset: X_train_2d={X_train_2d.shape}')

---## Part 1 — Calculus Foundations from ScratchWe build up from basic derivatives to the multi-variable calculusneeded for ML optimization.

### 1.1 Numerical DifferentiationThe derivative of $f$ at $x$ is defined as:$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$Three finite-difference approximations:- **Forward:** $\frac{f(x+h) - f(x)}{h}$ — $O(h)$ error- **Backward:** $\frac{f(x) - f(x-h)}{h}$ — $O(h)$ error- **Central:** $\frac{f(x+h) - f(x-h)}{2h}$ — $O(h^2)$ error (much better!)

In [ ]:
def forward_difference(
    f: callable,
    x: float,
    h: float = 1e-5,
) -> float:
    """Compute forward finite difference approximation to f'(x).

    Args:
        f: Scalar function.
        x: Point at which to evaluate the derivative.
        h: Step size.

    Returns:
        Approximate derivative.
    """
    return (f(x + h) - f(x)) / h


def central_difference(
    f: callable,
    x: float,
    h: float = 1e-5,
) -> float:
    """Compute central finite difference approximation to f'(x).

    Central differences are O(h²), much more accurate than forward/backward.

    Args:
        f: Scalar function.
        x: Point at which to evaluate the derivative.
        h: Step size.

    Returns:
        Approximate derivative.
    """
    return (f(x + h) - f(x - h)) / (2 * h)


# Test on known functions
test_functions = [
    ('x²',   lambda x: x ** 2,  lambda x: 2 * x),
    ('sin(x)', np.sin,          np.cos),
    ('eˣ',   np.exp,            np.exp),
    ('x³-2x', lambda x: x**3 - 2*x, lambda x: 3*x**2 - 2),
]

x_test = 1.5
print(f'Numerical differentiation at x = {x_test}\n')
results = []
for name, f, df_exact in test_functions:
    exact = df_exact(x_test)
    fwd = forward_difference(f, x_test)
    ctr = central_difference(f, x_test)
    results.append({
        'Function': name,
        'Exact': f'{exact:.8f}',
        'Forward': f'{fwd:.8f}',
        'Central': f'{ctr:.8f}',
        'Fwd Error': f'{abs(fwd - exact):.2e}',
        'Ctr Error': f'{abs(ctr - exact):.2e}',
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Let's visualize how step size $h$ affects the accuracy of finite differences.

In [ ]:
def analyze_step_size_effect() -> None:
    """Show error vs step size for forward and central differences."""
    f = lambda x: np.sin(x)
    df_exact = np.cos(1.5)  # True derivative at x=1.5

    h_values = np.logspace(-15, 0, 100)
    fwd_errors = [abs(forward_difference(f, 1.5, h) - df_exact) for h in h_values]
    ctr_errors = [abs(central_difference(f, 1.5, h) - df_exact) for h in h_values]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.loglog(h_values, fwd_errors, color=COLORS['blue'], linewidth=2,
               label='Forward Difference O(h)')
    ax.loglog(h_values, ctr_errors, color=COLORS['red'], linewidth=2,
               label='Central Difference O(h²)')

    # Theoretical error lines
    ax.loglog(h_values, h_values, '--', color=COLORS['gray'], alpha=0.5, label='O(h)')
    ax.loglog(h_values, h_values**2, '--', color=COLORS['orange'], alpha=0.5, label='O(h²)')

    ax.set_xlabel('Step Size h', fontsize=12)
    ax.set_ylabel('Absolute Error', fontsize=12)
    ax.set_title('Finite Difference Error vs Step Size\n'
                  'Sweet spot: too small → floating-point noise, too large → truncation error')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)
    ax.set_xlim(1e-15, 1)
    ax.set_ylim(1e-14, 10)
    plt.tight_layout()
    plt.show()

    # Find optimal h
    best_h_fwd = h_values[np.argmin(fwd_errors)]
    best_h_ctr = h_values[np.argmin(ctr_errors)]
    print(f'Optimal h for forward difference:  {best_h_fwd:.2e} (error: {min(fwd_errors):.2e})')
    print(f'Optimal h for central difference:  {best_h_ctr:.2e} (error: {min(ctr_errors):.2e})')


analyze_step_size_effect()

### 1.2 Partial Derivatives and GradientsFor a function $f: \mathbb{R}^d \to \mathbb{R}$, the **gradient** collectsall partial derivatives:$$\nabla f(\mathbf{x}) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \vdots \\ \frac{\partial f}{\partial x_d} \end{bmatrix}$$The gradient points in the direction of steepest ascent. Its magnitudetells us how steeply the function increases.

In [ ]:
def numerical_gradient(
    f: callable,
    x: np.ndarray,
    h: float = 1e-5,
) -> np.ndarray:
    """Compute gradient of f at x using central differences.

    Args:
        f: Scalar-valued function of a vector.
        x: Point at which to evaluate the gradient, shape (d,).
        h: Step size.

    Returns:
        Gradient vector of shape (d,).
    """
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += h
        x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad


# Example: quadratic bowl f(x, y) = x² + 2y²
def quadratic_bowl(x: np.ndarray) -> float:
    """Compute f(x, y) = x² + 2y².

    Args:
        x: Input vector of shape (2,).

    Returns:
        Scalar function value.
    """
    return x[0] ** 2 + 2 * x[1] ** 2


# Analytical gradient: [2x, 4y]
point = np.array([3.0, 2.0])
numerical_grad = numerical_gradient(quadratic_bowl, point)
analytical_grad = np.array([2 * point[0], 4 * point[1]])

print(f'Point: {point}')
print(f'f(x, y) = {quadratic_bowl(point):.4f}')
print(f'Numerical gradient:  {numerical_grad}')
print(f'Analytical gradient: {analytical_grad}')
print(f'Max difference:      {np.max(np.abs(numerical_grad - analytical_grad)):.2e}')
assert np.allclose(numerical_grad, analytical_grad, atol=1e-6), 'Gradient mismatch!'

Let's visualize the gradient field on a 2D loss surface.

In [ ]:
def visualize_gradient_field() -> None:
    """Visualize gradient vectors on a 2D loss surface."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Function surfaces
    functions = [
        ('$f(x,y) = x^2 + 2y^2$ (Bowl)', quadratic_bowl,
         lambda x: np.array([2 * x[0], 4 * x[1]])),
        ('$f(x,y) = \\sin(x) + \\cos(y)$', lambda x: np.sin(x[0]) + np.cos(x[1]),
         lambda x: np.array([np.cos(x[0]), -np.sin(x[1])])),
    ]

    for idx, (title, f, grad_f) in enumerate(functions):
        x_range = np.linspace(-3, 3, 100)
        y_range = np.linspace(-3, 3, 100)
        xx, yy = np.meshgrid(x_range, y_range)
        zz = np.array([[f(np.array([xi, yi])) for xi in x_range] for yi in y_range])

        axes[idx].contourf(xx, yy, zz, levels=30, cmap='viridis', alpha=0.8)
        axes[idx].contour(xx, yy, zz, levels=15, colors='white', linewidths=0.3)

        # Gradient arrows on a grid
        grid_x = np.linspace(-2.5, 2.5, 10)
        grid_y = np.linspace(-2.5, 2.5, 10)
        for gx in grid_x:
            for gy in grid_y:
                pt = np.array([gx, gy])
                g = grad_f(pt)
                g_norm = np.linalg.norm(g)
                if g_norm > 0.1:
                    # Normalize for display, negative = descent direction
                    g_unit = -g / g_norm * 0.3
                    axes[idx].arrow(gx, gy, g_unit[0], g_unit[1],
                                    head_width=0.08, head_length=0.04,
                                    fc=COLORS['red'], ec=COLORS['red'], alpha=0.6)

        axes[idx].set_xlabel('x', fontsize=12)
        axes[idx].set_ylabel('y', fontsize=12)
        axes[idx].set_title(title)
        axes[idx].set_aspect('equal')

    plt.suptitle('Negative Gradient Field (Descent Direction)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_gradient_field()

### 1.3 The Chain RuleThe **chain rule** is the mathematical foundation of backpropagation.For composed functions $f(g(x))$:$$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$For a computation graph with multiple variables:$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w}$$This is exactly how PyTorch autograd works — it chains local derivativesthrough the computation graph.

In [ ]:
def chain_rule_demo() -> None:
    """Demonstrate the chain rule with explicit computation graphs."""
    # Example: f(x) = sin(x²)
    # g(x) = x², f(g) = sin(g)
    # df/dx = cos(g) · 2x = cos(x²) · 2x

    x_vals = np.linspace(-3, 3, 300)

    # Forward pass
    g_vals = x_vals ** 2           # Inner function
    f_vals = np.sin(g_vals)        # Outer function

    # Chain rule: df/dx = df/dg · dg/dx
    df_dg = np.cos(g_vals)        # cos(x²)
    dg_dx = 2 * x_vals            # 2x
    chain_deriv = df_dg * dg_dx   # cos(x²) · 2x

    # Verify with numerical differentiation
    f_composed = lambda x: np.sin(x ** 2)
    numerical_deriv = np.array([central_difference(f_composed, x) for x in x_vals])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Function and derivative
    axes[0].plot(x_vals, f_vals, color=COLORS['blue'], linewidth=2, label='$f(x) = \sin(x^2)$')
    axes[0].plot(x_vals, chain_deriv, color=COLORS['red'], linewidth=2, label="$f'(x) = 2x\cos(x^2)$")
    axes[0].set_xlabel('x')
    axes[0].set_title('Function and Its Derivative')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.2)

    # Chain rule decomposition
    axes[1].plot(x_vals, df_dg, color=COLORS['green'], linewidth=2,
                  label='$\partial f/\partial g = \cos(x^2)$')
    axes[1].plot(x_vals, dg_dx, color=COLORS['orange'], linewidth=2,
                  label='$\partial g/\partial x = 2x$')
    axes[1].plot(x_vals, chain_deriv, '--', color=COLORS['red'], linewidth=2,
                  label='Product (chain rule)')
    axes[1].set_xlabel('x')
    axes[1].set_title('Chain Rule Decomposition')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.2)

    # Verification against numerical
    error = np.abs(chain_deriv - numerical_deriv)
    axes[2].semilogy(x_vals, error, color=COLORS['purple'], linewidth=1.5)
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('|Chain Rule - Numerical|')
    axes[2].set_title(f'Verification Error (max: {error.max():.2e})')
    axes[2].grid(True, alpha=0.2)

    plt.suptitle('Chain Rule: $d/dx[\sin(x^2)] = \cos(x^2) \cdot 2x$',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


chain_rule_demo()

### 1.4 Multi-Layer Chain Rule (Backpropagation Preview)In a neural network, we chain derivatives through multiple layers:$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial \mathbf{h}_2} \cdot \frac{\partial \mathbf{h}_2}{\partial \mathbf{h}_1} \cdot \frac{\partial \mathbf{h}_1}{\partial \mathbf{W}_1}$$Let's implement a simple 2-layer computation graph and manuallycompute gradients using the chain rule.

In [ ]:
def manual_backprop_demo() -> None:
    """Demonstrate manual backpropagation through a 2-layer network."""
    rng = np.random.RandomState(SEED)

    # Simple 2-layer network: x -> Linear -> ReLU -> Linear -> MSE Loss
    # Dimensions: input(3) -> hidden(4) -> output(1)
    x = rng.randn(3).astype(np.float64)
    y_true = np.array([1.0])

    W1 = rng.randn(4, 3).astype(np.float64) * 0.5
    b1 = np.zeros(4)
    W2 = rng.randn(1, 4).astype(np.float64) * 0.5
    b2 = np.zeros(1)

    # ── Forward pass ──
    z1 = W1 @ x + b1           # Pre-activation layer 1
    h1 = np.maximum(z1, 0)     # ReLU activation
    z2 = W2 @ h1 + b2          # Output layer
    y_pred = z2                 # No activation (regression)
    loss = 0.5 * np.sum((y_pred - y_true) ** 2)  # MSE

    print('=== Forward Pass ===')
    print(f'Input x:     {x}')
    print(f'z1 = W1·x:   {z1}')
    print(f'h1 = ReLU:   {h1}')
    print(f'y_pred:      {y_pred}')
    print(f'y_true:      {y_true}')
    print(f'Loss (MSE):  {loss:.6f}')

    # ── Backward pass (chain rule) ──
    # dL/dy_pred = y_pred - y_true
    dL_dy = y_pred - y_true

    # dL/dW2 = dL/dy · dy/dW2 = dL/dy · h1^T
    dL_dW2 = np.outer(dL_dy, h1)
    dL_db2 = dL_dy.copy()

    # dL/dh1 = W2^T · dL/dy
    dL_dh1 = W2.T @ dL_dy

    # dL/dz1 = dL/dh1 · ReLU'(z1)
    relu_grad = (z1 > 0).astype(np.float64)
    dL_dz1 = dL_dh1 * relu_grad

    # dL/dW1 = dL/dz1 · x^T
    dL_dW1 = np.outer(dL_dz1, x)
    dL_db1 = dL_dz1.copy()

    print('\n=== Backward Pass (Manual Chain Rule) ===')
    print(f'dL/dy_pred: {dL_dy}')
    print(f'dL/dW2 shape: {dL_dW2.shape}')
    print(f'dL/dW1 shape: {dL_dW1.shape}')

    # ── Verify against PyTorch autograd ──
    x_t = torch.tensor(x, dtype=torch.float64)
    y_t = torch.tensor(y_true, dtype=torch.float64)
    W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
    b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
    W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
    b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

    z1_t = W1_t @ x_t + b1_t
    h1_t = torch.relu(z1_t)
    y_pred_t = W2_t @ h1_t + b2_t
    loss_t = 0.5 * torch.sum((y_pred_t - y_t) ** 2)
    loss_t.backward()

    print('\n=== Verification Against PyTorch Autograd ===')
    print(f'dL/dW1 max diff: {np.max(np.abs(dL_dW1 - W1_t.grad.numpy())):.2e}')
    print(f'dL/db1 max diff: {np.max(np.abs(dL_db1 - b1_t.grad.numpy())):.2e}')
    print(f'dL/dW2 max diff: {np.max(np.abs(dL_dW2 - W2_t.grad.numpy())):.2e}')
    print(f'dL/db2 max diff: {np.max(np.abs(dL_db2 - b2_t.grad.numpy())):.2e}')

    assert np.allclose(dL_dW1, W1_t.grad.numpy(), atol=1e-10), 'W1 gradient mismatch!'
    assert np.allclose(dL_dW2, W2_t.grad.numpy(), atol=1e-10), 'W2 gradient mismatch!'
    print('All gradients match PyTorch autograd!')


manual_backprop_demo()

### 1.5 Jacobians and HessiansFor a vector-valued function $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$,the **Jacobian** is the matrix of all partial derivatives:$$\mathbf{J} = \begin{bmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n} \end{bmatrix}$$For a scalar function, the **Hessian** is the matrix of second derivatives:$$\mathbf{H} = \begin{bmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} \end{bmatrix}$$The Hessian eigenvalues reveal the curvature of the loss surface:- All positive → local minimum- All negative → local maximum- Mixed signs → saddle point

In [ ]:
def numerical_jacobian(
    f: callable,
    x: np.ndarray,
    m: int,
    h: float = 1e-5,
) -> np.ndarray:
    """Compute Jacobian of f: R^n -> R^m numerically.

    Args:
        f: Vector-valued function.
        x: Input point of shape (n,).
        m: Output dimension.
        h: Step size.

    Returns:
        Jacobian matrix of shape (m, n).
    """
    n = len(x)
    J = np.zeros((m, n))
    for j in range(n):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[j] += h
        x_minus[j] -= h
        J[:, j] = (f(x_plus) - f(x_minus)) / (2 * h)
    return J


def numerical_hessian(
    f: callable,
    x: np.ndarray,
    h: float = 1e-4,
) -> np.ndarray:
    """Compute Hessian of scalar function f numerically.

    Args:
        f: Scalar-valued function.
        x: Input point of shape (n,).
        h: Step size.

    Returns:
        Hessian matrix of shape (n, n).
    """
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            x_pp = x.copy(); x_pp[i] += h; x_pp[j] += h
            x_pm = x.copy(); x_pm[i] += h; x_pm[j] -= h
            x_mp = x.copy(); x_mp[i] -= h; x_mp[j] += h
            x_mm = x.copy(); x_mm[i] -= h; x_mm[j] -= h
            H[i, j] = (f(x_pp) - f(x_pm) - f(x_mp) + f(x_mm)) / (4 * h * h)
    return H


# Jacobian example: f(x, y) = [x²y, 5x + sin(y)]
def vector_func(x: np.ndarray) -> np.ndarray:
    """Example vector function f: R² -> R².

    Args:
        x: Input of shape (2,).

    Returns:
        Output of shape (2,).
    """
    return np.array([x[0]**2 * x[1], 5 * x[0] + np.sin(x[1])])


point_j = np.array([1.0, 2.0])
J_num = numerical_jacobian(vector_func, point_j, m=2)

# Analytical Jacobian: [[2xy, x²], [5, cos(y)]]
J_exact = np.array([
    [2 * point_j[0] * point_j[1], point_j[0] ** 2],
    [5.0, np.cos(point_j[1])],
])

print('=== Jacobian ===')
print(f'Point: {point_j}')
print(f'Numerical:\n{J_num}')
print(f'Analytical:\n{J_exact}')
print(f'Max diff: {np.max(np.abs(J_num - J_exact)):.2e}')
assert np.allclose(J_num, J_exact, atol=1e-5), 'Jacobian mismatch!'

# Hessian example
point_h = np.array([1.0, 2.0])
H_num = numerical_hessian(quadratic_bowl, point_h)
# Analytical Hessian for x² + 2y²: [[2, 0], [0, 4]]
H_exact = np.array([[2.0, 0.0], [0.0, 4.0]])

print('\n=== Hessian ===')
print(f'Numerical:\n{H_num}')
print(f'Analytical:\n{H_exact}')
print(f'Max diff: {np.max(np.abs(H_num - H_exact)):.2e}')
assert np.allclose(H_num, H_exact, atol=1e-3), 'Hessian mismatch!'

### 1.6 Verification Against PyTorch AutogradPyTorch's autograd system computes exact gradients via reverse-modeautomatic differentiation (not finite differences). Let's verify ournumerical computations match autograd.

In [ ]:
def verify_against_autograd() -> None:
    """Compare numerical gradients, Jacobians against PyTorch autograd."""
    # Gradient verification
    x_np = np.array([2.0, 3.0, -1.0])

    def complex_scalar_func(x: np.ndarray) -> float:
        """Compute f(x) = sum(sin(x) * x²).

        Args:
            x: Input vector.

        Returns:
            Scalar output.
        """
        return np.sum(np.sin(x) * x ** 2)

    grad_numerical = numerical_gradient(complex_scalar_func, x_np)

    # PyTorch autograd
    x_t = torch.tensor(x_np, requires_grad=True)
    y_t = torch.sum(torch.sin(x_t) * x_t ** 2)
    y_t.backward()
    grad_autograd = x_t.grad.numpy()

    print('=== Gradient Comparison ===')
    print(f'Numerical:  {grad_numerical}')
    print(f'Autograd:   {grad_autograd}')
    print(f'Max diff:   {np.max(np.abs(grad_numerical - grad_autograd)):.2e}')
    assert np.allclose(grad_numerical, grad_autograd, atol=1e-5), 'Gradient mismatch!'
    print('PASSED!\n')

    # Jacobian verification
    def softmax_func(x: np.ndarray) -> np.ndarray:
        """Compute softmax.

        Args:
            x: Input logits.

        Returns:
            Softmax output.
        """
        exp_x = np.exp(x - np.max(x))
        return exp_x / exp_x.sum()

    logits_np = np.array([1.0, 2.0, 3.0])
    J_numerical = numerical_jacobian(softmax_func, logits_np, m=3)

    # PyTorch Jacobian
    logits_t = torch.tensor(logits_np, requires_grad=True)
    J_autograd = torch.autograd.functional.jacobian(
        lambda x: torch.softmax(x, dim=0), logits_t,
    ).numpy()

    print('=== Jacobian Comparison (Softmax) ===')
    print(f'Max diff: {np.max(np.abs(J_numerical - J_autograd)):.2e}')
    assert np.allclose(J_numerical, J_autograd, atol=1e-4), 'Jacobian mismatch!'
    print('PASSED!')

    # Property: softmax Jacobian rows sum to 0
    row_sums = J_autograd.sum(axis=1)
    print(f'Row sums (should be ~0): {row_sums}')


verify_against_autograd()

### 1.7 Gradient Descent from Scratch**Gradient descent** iteratively updates parameters by moving in thenegative gradient direction:$$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t)$$where $\eta$ is the **learning rate**.

In [ ]:
def gradient_descent(
    f: callable,
    grad_f: callable,
    x0: np.ndarray,
    learning_rate: float,
    n_iterations: int,
) -> tuple[np.ndarray, list[float], list[np.ndarray]]:
    """Run vanilla gradient descent.

    Args:
        f: Objective function to minimize.
        grad_f: Gradient function.
        x0: Initial parameter values.
        learning_rate: Step size.
        n_iterations: Number of iterations.

    Returns:
        Tuple of (final_params, loss_history, param_history).
    """
    x = x0.copy()
    loss_history: list[float] = [f(x)]
    param_history: list[np.ndarray] = [x.copy()]

    for _ in range(n_iterations):
        g = grad_f(x)
        x = x - learning_rate * g
        loss_history.append(f(x))
        param_history.append(x.copy())

    return x, loss_history, param_history


# Minimize x² + 2y²
grad_bowl = lambda x: np.array([2 * x[0], 4 * x[1]])
x0 = np.array([4.0, 3.0])

x_final, losses, params = gradient_descent(
    quadratic_bowl, grad_bowl, x0, learning_rate=0.1, n_iterations=50,
)

print(f'Start: {x0}, f = {quadratic_bowl(x0):.4f}')
print(f'Final: [{x_final[0]:.6f}, {x_final[1]:.6f}], f = {quadratic_bowl(x_final):.6f}')

# Visualize path
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contour plot with path
x_range = np.linspace(-5, 5, 100)
y_range = np.linspace(-4, 4, 100)
xx, yy = np.meshgrid(x_range, y_range)
zz = xx ** 2 + 2 * yy ** 2

axes[0].contourf(xx, yy, zz, levels=30, cmap='viridis', alpha=0.8)
axes[0].contour(xx, yy, zz, levels=15, colors='white', linewidths=0.3)
params_arr = np.array(params)
axes[0].plot(params_arr[:, 0], params_arr[:, 1], 'o-', color=COLORS['red'],
              markersize=3, linewidth=1.5, label='GD path')
axes[0].plot(x0[0], x0[1], 'r*', markersize=15, label='Start')
axes[0].plot(0, 0, 'g*', markersize=15, label='Minimum')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Gradient Descent Path')
axes[0].legend()

# Loss curve
axes[1].plot(losses, color=COLORS['blue'], linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Curve')
axes[1].grid(True, alpha=0.2)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

### 1.8 Learning Rate EffectsThe learning rate $\eta$ is the most critical hyperparameter:- **Too small:** convergence is painfully slow- **Too large:** oscillation or divergence- **Just right:** smooth, fast convergence

In [ ]:
def learning_rate_comparison() -> None:
    """Show effect of different learning rates on convergence."""
    learning_rates = [0.001, 0.01, 0.1, 0.3, 0.5, 0.9]
    x0 = np.array([4.0, 3.0])
    n_iters = 50

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))

    for idx, lr in enumerate(learning_rates):
        ax = axes.ravel()[idx]

        try:
            _, losses, params = gradient_descent(
                quadratic_bowl, grad_bowl, x0, lr, n_iters,
            )
            params_arr = np.array(params)

            # Check for divergence
            diverged = any(l > 1e6 for l in losses)
        except (OverflowError, FloatingPointError):
            diverged = True

        # Contour background
        x_r = np.linspace(-6, 6, 80)
        y_r = np.linspace(-5, 5, 80)
        xx_lr, yy_lr = np.meshgrid(x_r, y_r)
        zz_lr = xx_lr ** 2 + 2 * yy_lr ** 2
        ax.contourf(xx_lr, yy_lr, zz_lr, levels=20, cmap='viridis', alpha=0.7)

        if not diverged:
            ax.plot(params_arr[:, 0], params_arr[:, 1], 'o-',
                     color=COLORS['red'], markersize=2, linewidth=1)
            status = f'Final loss: {losses[-1]:.4f}'
        else:
            status = 'DIVERGED!'

        ax.plot(x0[0], x0[1], 'r*', markersize=12)
        ax.plot(0, 0, 'g*', markersize=12)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(f'LR = {lr}\n{status}')
        ax.set_xlim(-6, 6)
        ax.set_ylim(-5, 5)

    plt.suptitle('Effect of Learning Rate on Gradient Descent',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


learning_rate_comparison()

### 1.9 Gradient Descent with Momentum**Momentum** accelerates convergence by accumulating a velocity vector:$$\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla \mathcal{L}(\theta_t)$$$$\theta_{t+1} = \theta_t - \eta \mathbf{v}_{t+1}$$This helps:- Navigate ravines (elongated loss surfaces) faster- Smooth out noisy gradients- Escape shallow local minima

In [ ]:
def gradient_descent_momentum(
    f: callable,
    grad_f: callable,
    x0: np.ndarray,
    learning_rate: float,
    n_iterations: int,
    beta: float = 0.9,
) -> tuple[np.ndarray, list[float], list[np.ndarray]]:
    """Run gradient descent with momentum.

    Args:
        f: Objective function.
        grad_f: Gradient function.
        x0: Initial parameters.
        learning_rate: Step size.
        n_iterations: Number of iterations.
        beta: Momentum coefficient (0 = no momentum).

    Returns:
        Tuple of (final_params, loss_history, param_history).
    """
    x = x0.copy()
    v = np.zeros_like(x)
    loss_history: list[float] = [f(x)]
    param_history: list[np.ndarray] = [x.copy()]

    for _ in range(n_iterations):
        g = grad_f(x)
        v = beta * v + g
        x = x - learning_rate * v
        loss_history.append(f(x))
        param_history.append(x.copy())

    return x, loss_history, param_history


# Compare vanilla GD vs momentum on an elongated surface
# Rosenbrock-like: f(x,y) = (1-x)² + 100(y-x²)² (narrow curved valley)
def rosenbrock(x: np.ndarray) -> float:
    """Compute Rosenbrock function (banana function).

    Args:
        x: Input of shape (2,).

    Returns:
        Scalar function value.
    """
    return (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2


def rosenbrock_grad(x: np.ndarray) -> np.ndarray:
    """Compute gradient of Rosenbrock function.

    Args:
        x: Input of shape (2,).

    Returns:
        Gradient of shape (2,).
    """
    dx = -2 * (1 - x[0]) + 200 * (x[1] - x[0] ** 2) * (-2 * x[0])
    dy = 200 * (x[1] - x[0] ** 2)
    return np.array([dx, dy])


x0_rosen = np.array([-1.0, 1.0])
lr_rosen = 0.001
n_iters_rosen = 5000

_, losses_gd, params_gd = gradient_descent(
    rosenbrock, rosenbrock_grad, x0_rosen, lr_rosen, n_iters_rosen,
)
_, losses_mom, params_mom = gradient_descent_momentum(
    rosenbrock, rosenbrock_grad, x0_rosen, lr_rosen, n_iters_rosen, beta=0.9,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Path on contour
x_r = np.linspace(-2, 2, 200)
y_r = np.linspace(-1, 3, 200)
xx_r, yy_r = np.meshgrid(x_r, y_r)
zz_r = (1 - xx_r) ** 2 + 100 * (yy_r - xx_r ** 2) ** 2

axes[0].contourf(xx_r, yy_r, np.log10(zz_r + 1), levels=30, cmap='viridis', alpha=0.8)
gd_arr = np.array(params_gd)
mom_arr = np.array(params_mom)
# Plot every 50th point for clarity
axes[0].plot(gd_arr[::50, 0], gd_arr[::50, 1], 'o-', color=COLORS['blue'],
              markersize=3, linewidth=1, label='Vanilla GD', alpha=0.8)
axes[0].plot(mom_arr[::50, 0], mom_arr[::50, 1], 's-', color=COLORS['red'],
              markersize=3, linewidth=1, label='Momentum', alpha=0.8)
axes[0].plot(1, 1, 'g*', markersize=15, label='Minimum (1,1)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Rosenbrock Function: GD vs Momentum')
axes[0].legend(fontsize=9)

# Loss curves
axes[1].semilogy(losses_gd, color=COLORS['blue'], linewidth=2, label='Vanilla GD', alpha=0.8)
axes[1].semilogy(losses_mom, color=COLORS['red'], linewidth=2, label='Momentum', alpha=0.8)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Convergence Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f'Final loss — Vanilla GD:  {losses_gd[-1]:.6f}')
print(f'Final loss — Momentum:    {losses_mom[-1]:.6f}')

---## Part 2 — Putting It All Together: Optimizer ClassLet's assemble our optimization methods into a reusable optimizer classthat mimics PyTorch's optimizer API.

In [ ]:
class GradientOptimizer:
    """From-scratch gradient-based optimizer supporting multiple methods.

    Supports vanilla SGD, SGD with momentum, and a simple adaptive
    learning rate method (RMSProp-like).

    Attributes:
        params: Current parameter values.
        learning_rate: Step size.
        method: Optimization method name.
        loss_history: Recorded loss values.
        param_history: Recorded parameter snapshots.
    """

    def __init__(
        self,
        params: np.ndarray,
        learning_rate: float = 0.01,
        method: str = 'sgd',
        beta: float = 0.9,
        epsilon: float = 1e-8,
    ) -> None:
        """Initialize optimizer.

        Args:
            params: Initial parameter values.
            learning_rate: Step size.
            method: One of 'sgd', 'momentum', 'rmsprop'.
            beta: Momentum/decay coefficient.
            epsilon: Small constant for numerical stability.
        """
        self.params = params.copy()
        self.learning_rate = learning_rate
        self.method = method
        self.beta = beta
        self.epsilon = epsilon

        self.velocity = np.zeros_like(params)
        self.cache = np.zeros_like(params)

        self.loss_history: list[float] = []
        self.param_history: list[np.ndarray] = [params.copy()]

    def step(self, gradient: np.ndarray, loss: float) -> None:
        """Perform one optimization step.

        Args:
            gradient: Gradient of loss w.r.t. parameters.
            loss: Current loss value (for recording).
        """
        self.loss_history.append(loss)

        if self.method == 'sgd':
            self.params -= self.learning_rate * gradient

        elif self.method == 'momentum':
            self.velocity = self.beta * self.velocity + gradient
            self.params -= self.learning_rate * self.velocity

        elif self.method == 'rmsprop':
            self.cache = self.beta * self.cache + (1 - self.beta) * gradient ** 2
            self.params -= self.learning_rate * gradient / (
                np.sqrt(self.cache) + self.epsilon
            )

        self.param_history.append(self.params.copy())


# Sanity check: optimize the quadratic bowl with each method
x0_test = np.array([4.0, 3.0])
methods = ['sgd', 'momentum', 'rmsprop']
results = []

for method_name in methods:
    opt = GradientOptimizer(x0_test, learning_rate=0.05, method=method_name)
    for _ in range(100):
        loss_val = quadratic_bowl(opt.params)
        grad_val = grad_bowl(opt.params)
        opt.step(grad_val, loss_val)

    final_loss = quadratic_bowl(opt.params)
    results.append({
        'Method': method_name.upper(),
        'Final x': f'[{opt.params[0]:.6f}, {opt.params[1]:.6f}]',
        'Final Loss': f'{final_loss:.8f}',
        'Steps to < 0.01': next(
            (i for i, l in enumerate(opt.loss_history) if l < 0.01), len(opt.loss_history)
        ),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

---## Part 3 — Training on California HousingWe apply gradient descent to train a linear regression model on realdata, comparing our from-scratch optimizer against closed-form andlibrary solutions.

### 3.1 Linear Regression via Gradient Descent

In [ ]:
def mse_loss(X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float) -> float:
    """Compute mean squared error loss.

    Args:
        X: Feature matrix of shape (n, d).
        y: Target values of shape (n,).
        w: Weight vector of shape (d,).
        b: Bias scalar.

    Returns:
        MSE loss value.
    """
    predictions = X @ w + b
    return np.mean((predictions - y) ** 2)


def mse_gradient(
    X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float,
) -> tuple[np.ndarray, float]:
    """Compute gradient of MSE w.r.t. weights and bias.

    Args:
        X: Feature matrix of shape (n, d).
        y: Target values of shape (n,).
        w: Weight vector of shape (d,).
        b: Bias scalar.

    Returns:
        Tuple of (dw, db) — gradients for weights and bias.
    """
    n = len(y)
    predictions = X @ w + b
    residuals = predictions - y
    dw = (2.0 / n) * X.T @ residuals
    db = (2.0 / n) * residuals.sum()
    return dw, db


def train_linear_regression_gd(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    learning_rate: float = 0.01,
    n_iterations: int = 500,
    method: str = 'sgd',
) -> tuple[np.ndarray, float, list[float], list[float]]:
    """Train linear regression using gradient descent.

    Args:
        X_train: Training features.
        y_train: Training targets.
        X_test: Test features.
        y_test: Test targets.
        learning_rate: Step size.
        n_iterations: Number of GD iterations.
        method: Optimizer method.

    Returns:
        Tuple of (weights, bias, train_losses, test_losses).
    """
    d = X_train.shape[1]
    w = np.zeros(d)
    b = 0.0

    opt_w = GradientOptimizer(w, learning_rate, method)
    train_losses: list[float] = []
    test_losses: list[float] = []

    for _ in range(n_iterations):
        loss_train = mse_loss(X_train, y_train, opt_w.params, b)
        loss_test = mse_loss(X_test, y_test, opt_w.params, b)
        train_losses.append(loss_train)
        test_losses.append(loss_test)

        dw, db = mse_gradient(X_train, y_train, opt_w.params, b)
        opt_w.step(dw, loss_train)
        b -= learning_rate * db

    return opt_w.params, b, train_losses, test_losses


# Train with different methods
method_results = {}
for method_name in ['sgd', 'momentum', 'rmsprop']:
    w, b, train_l, test_l = train_linear_regression_gd(
        X_train, y_train, X_test, y_test,
        learning_rate=0.01, n_iterations=500, method=method_name,
    )
    method_results[method_name] = (w, b, train_l, test_l)
    print(f'{method_name.upper():10s}: Final Train MSE = {train_l[-1]:.4f}, '
          f'Test MSE = {test_l[-1]:.4f}')

### 3.2 Comparison with Closed-Form and Library Solutions

In [ ]:
def compare_solutions() -> None:
    """Compare GD solution against closed-form and sklearn."""
    # Closed-form: w = (X^T X)^{-1} X^T y
    X_bias = np.column_stack([X_train, np.ones(len(X_train))])
    w_closed = np.linalg.lstsq(X_bias, y_train, rcond=None)[0]
    w_cf, b_cf = w_closed[:-1], w_closed[-1]
    mse_closed_train = mse_loss(X_train, y_train, w_cf, b_cf)
    mse_closed_test = mse_loss(X_test, y_test, w_cf, b_cf)

    # sklearn
    from sklearn.linear_model import LinearRegression
    lr_sk = LinearRegression()
    lr_sk.fit(X_train, y_train)
    mse_sk_train = np.mean((lr_sk.predict(X_train) - y_train) ** 2)
    mse_sk_test = np.mean((lr_sk.predict(X_test) - y_test) ** 2)

    # Summary table
    rows = []
    for name, (w, b, trl, tel) in method_results.items():
        rows.append({
            'Method': f'GD ({name})',
            'Train MSE': f'{trl[-1]:.4f}',
            'Test MSE': f'{tel[-1]:.4f}',
        })
    rows.append({'Method': 'Closed-Form', 'Train MSE': f'{mse_closed_train:.4f}',
                 'Test MSE': f'{mse_closed_test:.4f}'})
    rows.append({'Method': 'sklearn', 'Train MSE': f'{mse_sk_train:.4f}',
                 'Test MSE': f'{mse_sk_test:.4f}'})

    compare_df = pd.DataFrame(rows)
    print(compare_df.to_string(index=False))

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, (_, _, trl, tel) in method_results.items():
        color = {'sgd': COLORS['blue'], 'momentum': COLORS['red'],
                 'rmsprop': COLORS['green']}[name]
        axes[0].plot(trl, color=color, linewidth=2, label=f'{name} (train)')
        axes[1].plot(tel, color=color, linewidth=2, label=f'{name} (test)')

    axes[0].axhline(mse_closed_train, color=COLORS['gray'], linestyle='--',
                     label=f'Closed-form: {mse_closed_train:.4f}')
    axes[1].axhline(mse_closed_test, color=COLORS['gray'], linestyle='--',
                     label=f'Closed-form: {mse_closed_test:.4f}')

    for ax, title in zip(axes, ['Train Loss', 'Test Loss']):
        ax.set_xlabel('Iteration')
        ax.set_ylabel('MSE')
        ax.set_title(title)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.2)

    plt.suptitle('Linear Regression: GD vs Closed-Form',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


compare_solutions()

---## Part 4 — Evaluation & Analysis

### 4.1 Loss Landscape VisualizationLet's visualize the loss landscape for our 2D linear regressionproblem and see how different optimizers navigate it.

In [ ]:
def visualize_2d_loss_landscape() -> None:
    """Visualize MSE loss surface for 2-feature linear regression."""
    # Train with 2D features for visualization
    w_range = np.linspace(-2, 6, 100)
    b_range = np.linspace(-2, 6, 100)

    # Fix w2 at optimal, vary w1 and bias
    X_bias_2d = np.column_stack([X_train_2d, np.ones(len(X_train_2d))])
    w_opt_2d = np.linalg.lstsq(X_bias_2d, y_train_2d, rcond=None)[0]

    # Compute loss surface over w1 and w2 (bias fixed at optimal)
    w1_range = np.linspace(w_opt_2d[0] - 3, w_opt_2d[0] + 3, 80)
    w2_range = np.linspace(w_opt_2d[1] - 3, w_opt_2d[1] + 3, 80)
    ww1, ww2 = np.meshgrid(w1_range, w2_range)
    loss_surface = np.zeros_like(ww1)

    for i in range(len(w2_range)):
        for j in range(len(w1_range)):
            w_test = np.array([ww1[i, j], ww2[i, j]])
            loss_surface[i, j] = mse_loss(X_train_2d, y_train_2d, w_test, w_opt_2d[2])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 2D contour
    axes[0].contourf(ww1, ww2, loss_surface, levels=30, cmap='viridis', alpha=0.8)
    axes[0].contour(ww1, ww2, loss_surface, levels=15, colors='white', linewidths=0.3)
    axes[0].plot(w_opt_2d[0], w_opt_2d[1], 'r*', markersize=15, label='Optimal')
    axes[0].set_xlabel('$w_1$ (MedInc weight)')
    axes[0].set_ylabel('$w_2$ (HouseAge weight)')
    axes[0].set_title('MSE Loss Surface')
    axes[0].legend()

    # 3D surface
    from mpl_toolkits.mplot3d import Axes3D
    ax3d = fig.add_subplot(122, projection='3d')
    ax3d.plot_surface(ww1, ww2, loss_surface, cmap='viridis', alpha=0.8,
                       edgecolor='none')
    ax3d.set_xlabel('$w_1$')
    ax3d.set_ylabel('$w_2$')
    ax3d.set_zlabel('MSE')
    ax3d.set_title('3D Loss Surface')

    # Remove the 2D axes[1] since we replaced it with 3D
    axes[1].remove()

    plt.suptitle('Linear Regression Loss Landscape (2 Features)',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Hessian at optimal
    def loss_func_2d(w: np.ndarray) -> float:
        """MSE loss as function of 2D weights only.

        Args:
            w: Weight vector of shape (2,).

        Returns:
            MSE loss.
        """
        return mse_loss(X_train_2d, y_train_2d, w, w_opt_2d[2])

    H = numerical_hessian(loss_func_2d, w_opt_2d[:2])
    eigvals = np.linalg.eigvalsh(H)
    condition_number = eigvals.max() / eigvals.min()

    print(f'Hessian eigenvalues at minimum: {eigvals}')
    print(f'Condition number: {condition_number:.2f}')
    print(f'High condition number → elongated loss surface → slow convergence for vanilla GD')


visualize_2d_loss_landscape()

### 4.2 Saddle Points and Local MinimaIn high dimensions, saddle points (mixed Hessian eigenvalues) are farmore common than local minima. Let's visualize both phenomena.

In [ ]:
def saddle_point_analysis() -> None:
    """Visualize saddle points and local minima."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    x_range = np.linspace(-3, 3, 200)
    y_range = np.linspace(-3, 3, 200)
    xx, yy = np.meshgrid(x_range, y_range)

    # 1. Saddle point: f(x,y) = x² - y²
    zz_saddle = xx ** 2 - yy ** 2
    axes[0].contourf(xx, yy, zz_saddle, levels=30, cmap='coolwarm', alpha=0.8)
    axes[0].contour(xx, yy, zz_saddle, levels=[0], colors='black', linewidths=2)
    axes[0].plot(0, 0, 'ko', markersize=10, label='Saddle point')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title('$f(x,y) = x^2 - y^2$\nSaddle Point')
    axes[0].legend()

    # Hessian at saddle: [[2, 0], [0, -2]] → eigenvalues: {2, -2}
    H_saddle = np.array([[2, 0], [0, -2]])
    eigs_saddle = np.linalg.eigvalsh(H_saddle)
    axes[0].set_xlabel(f'Hessian eigenvalues: {eigs_saddle}')

    # 2. Local minima: f(x,y) = sin(x)·sin(y) + 0.1(x²+y²)
    zz_local = np.sin(xx) * np.sin(yy) + 0.1 * (xx ** 2 + yy ** 2)
    axes[1].contourf(xx, yy, zz_local, levels=30, cmap='viridis', alpha=0.8)

    # Find local minima by running GD from multiple starting points
    rng = np.random.RandomState(SEED)
    f_local = lambda x: np.sin(x[0]) * np.sin(x[1]) + 0.1 * (x[0] ** 2 + x[1] ** 2)
    grad_local = lambda x: np.array([
        np.cos(x[0]) * np.sin(x[1]) + 0.2 * x[0],
        np.sin(x[0]) * np.cos(x[1]) + 0.2 * x[1],
    ])

    minima_found = []
    for _ in range(20):
        x0_rand = rng.uniform(-3, 3, size=2)
        x_min, _, _ = gradient_descent(f_local, grad_local, x0_rand, 0.05, 200)
        minima_found.append(x_min)

    minima_arr = np.array(minima_found)
    axes[1].scatter(minima_arr[:, 0], minima_arr[:, 1], c='red', s=50,
                     marker='*', zorder=5, label='Converged points')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].set_title('Multiple Local Minima')
    axes[1].legend(fontsize=9)

    # 3. GD getting stuck at saddle
    f_saddle = lambda x: x[0] ** 2 - x[1] ** 2
    grad_saddle = lambda x: np.array([2 * x[0], -2 * x[1]])

    # Start exactly at saddle (GD stays stuck)
    x0_exact = np.array([0.0, 0.0])
    x_result_exact, _, params_exact = gradient_descent(
        f_saddle, grad_saddle, x0_exact, 0.1, 50,
    )

    # Start with tiny perturbation (GD escapes)
    x0_perturb = np.array([0.001, 0.001])
    x_result_perturb, _, params_perturb = gradient_descent(
        f_saddle, grad_saddle, x0_perturb, 0.1, 50,
    )

    pp_exact = np.array(params_exact)
    pp_perturb = np.array(params_perturb)

    axes[2].contourf(xx, yy, zz_saddle, levels=30, cmap='coolwarm', alpha=0.7)
    axes[2].plot(pp_exact[:, 0], pp_exact[:, 1], 'ko-', markersize=3,
                  label='Start at (0,0): stuck')
    axes[2].plot(pp_perturb[:, 0], pp_perturb[:, 1], 'r^-', markersize=3,
                  label='Start at (ε,ε): escapes')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('y')
    axes[2].set_title('Escaping a Saddle Point')
    axes[2].legend(fontsize=9)

    plt.suptitle('Saddle Points and Local Minima', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


saddle_point_analysis()

### 4.3 Error Analysis: When Gradient Descent FailsLet's examine specific failure modes and common mistakes in gradient-basedoptimization.

In [ ]:
def error_analysis_optimization() -> None:
    """Analyze failure modes of gradient descent."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Failure 1: Feature scaling matters
    rng = np.random.RandomState(SEED)
    n = 200
    # Unscaled: feature 1 in [0, 1000], feature 2 in [0, 1]
    X_unscaled = np.column_stack([rng.uniform(0, 1000, n), rng.uniform(0, 1, n)])
    w_true = np.array([0.5, 3.0])
    y_unscaled = X_unscaled @ w_true + rng.randn(n) * 0.5

    # GD on unscaled data
    w_init = np.zeros(2)
    lr_test = 1e-7  # Need tiny LR for unscaled
    losses_unscaled = []
    w_curr = w_init.copy()
    for _ in range(200):
        pred = X_unscaled @ w_curr
        loss_val = np.mean((pred - y_unscaled) ** 2)
        losses_unscaled.append(loss_val)
        grad = (2.0 / n) * X_unscaled.T @ (pred - y_unscaled)
        w_curr = w_curr - lr_test * grad

    # GD on scaled data
    scaler_test = StandardScaler()
    X_scaled_test = scaler_test.fit_transform(X_unscaled)
    losses_scaled = []
    w_curr = w_init.copy()
    lr_test2 = 0.01
    for _ in range(200):
        pred = X_scaled_test @ w_curr
        loss_val = np.mean((pred - y_unscaled) ** 2)
        losses_scaled.append(loss_val)
        grad = (2.0 / n) * X_scaled_test.T @ (pred - y_unscaled)
        w_curr = w_curr - lr_test2 * grad

    axes[0].semilogy(losses_unscaled, color=COLORS['red'], linewidth=2, label='Unscaled')
    axes[0].semilogy(losses_scaled, color=COLORS['green'], linewidth=2, label='Scaled')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('MSE (log)')
    axes[0].set_title('Failure 1: Feature Scaling')
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)

    # Failure 2: Vanishing gradients (deep chain rule)
    depths = list(range(1, 30))
    grad_products = []
    for depth in depths:
        # Chain of sigmoid derivatives (max 0.25 each)
        grad_product = 0.25 ** depth
        grad_products.append(grad_product)

    axes[1].semilogy(depths, grad_products, 'o-', color=COLORS['blue'], linewidth=2)
    axes[1].set_xlabel('Network Depth (layers)')
    axes[1].set_ylabel('Gradient Magnitude')
    axes[1].set_title('Failure 2: Vanishing Gradients\n'
                       'sigmoid derivative max = 0.25')
    axes[1].grid(True, alpha=0.2)
    axes[1].axhline(1e-10, color=COLORS['red'], linestyle='--', alpha=0.5,
                     label='Below float32 precision')
    axes[1].legend()

    # Failure 3: Gradient explosion (too large LR)
    lrs_explosion = [0.1, 0.3, 0.5, 0.7, 0.9]
    for lr_exp in lrs_explosion:
        losses_exp = []
        w_exp = np.array([4.0, 3.0])
        for _ in range(30):
            loss_val = quadratic_bowl(w_exp)
            if loss_val > 1e10:
                losses_exp.append(1e10)
                continue
            losses_exp.append(loss_val)
            g = grad_bowl(w_exp)
            w_exp = w_exp - lr_exp * g
        axes[2].semilogy(losses_exp, linewidth=2, label=f'LR={lr_exp}')

    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Loss (log)')
    axes[2].set_title('Failure 3: Learning Rate Too Large')
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.2)

    plt.suptitle('Error Analysis: Gradient Descent Failure Modes',
                  fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Key failure modes:')
    print('  1. Unscaled features → extremely ill-conditioned loss surface')
    print('  2. Deep sigmoid networks → vanishing gradients (solved by ReLU/ResNets)')
    print('  3. Learning rate too large → oscillation or divergence')


error_analysis_optimization()

### 4.4 Convergence Rate Comparison

In [ ]:
import time


def convergence_benchmark() -> None:
    """Benchmark convergence of different optimization methods."""
    methods_bench = ['sgd', 'momentum', 'rmsprop']
    results = []

    for method_name in methods_bench:
        start = time.perf_counter()
        w, b, train_l, test_l = train_linear_regression_gd(
            X_train, y_train, X_test, y_test,
            learning_rate=0.01, n_iterations=500, method=method_name,
        )
        elapsed = time.perf_counter() - start

        # Find iteration where loss < threshold
        threshold = train_l[-1] * 1.01  # Within 1% of final
        converged_at = next(
            (i for i, l in enumerate(train_l) if l < threshold),
            len(train_l),
        )

        results.append({
            'Method': method_name.upper(),
            'Final Train MSE': f'{train_l[-1]:.4f}',
            'Final Test MSE': f'{test_l[-1]:.4f}',
            'Converged At': converged_at,
            'Time (ms)': f'{elapsed * 1000:.1f}',
        })

    # Add closed-form timing
    start = time.perf_counter()
    X_bias = np.column_stack([X_train, np.ones(len(X_train))])
    w_cf = np.linalg.lstsq(X_bias, y_train, rcond=None)[0]
    elapsed_cf = time.perf_counter() - start
    mse_cf = mse_loss(X_train, y_train, w_cf[:-1], w_cf[-1])
    mse_cf_test = mse_loss(X_test, y_test, w_cf[:-1], w_cf[-1])
    results.append({
        'Method': 'CLOSED-FORM',
        'Final Train MSE': f'{mse_cf:.4f}',
        'Final Test MSE': f'{mse_cf_test:.4f}',
        'Converged At': 1,
        'Time (ms)': f'{elapsed_cf * 1000:.1f}',
    })

    bench_df = pd.DataFrame(results)
    print(bench_df.to_string(index=False))


convergence_benchmark()

---## Part 5 — Summary & Lessons Learned### Key Takeaways- **Numerical differentiation** with central differences achieves $O(h^2)$  accuracy but suffers from floating-point noise at very small $h$. The  optimal $h \approx 10^{-5}$ balances truncation and rounding errors.- **The gradient** $\nabla f$ points in the direction of steepest ascent.  Gradient descent moves in the negative gradient direction to minimize  the loss. Every ML optimization algorithm is built on this principle.- **The chain rule** enables backpropagation by decomposing complex  derivatives into products of simple local derivatives. This is exactly  what PyTorch autograd automates.- **Momentum** and **adaptive learning rates** (RMSProp, Adam) dramatically  improve convergence on ill-conditioned loss surfaces compared to vanilla  gradient descent.- **Failure modes** — vanishing gradients, exploding gradients, saddle points,  and feature scaling issues — are real obstacles that motivate architectural  choices (ReLU, BatchNorm, skip connections) and hyperparameter tuning  throughout deep learning.### What's Next→ **01-10 (Computational Thinking & Complexity)** extends these ideas toanalyze the computational cost of the algorithms we've built, coveringBig-O analysis, vectorization, and memory hierarchy.